# Lab 05 — Bird's Eye View (Real-World Application)

A **bird's eye view** (also called **top-down view** or **inverse perspective mapping**) transforms an angled camera image into a flat overhead map. It's the backbone of:

- **ADAS / autonomous driving** — lane detection, free space estimation, distance to obstacles
- **Sports analytics** — player positions mapped to a 2D field for heat maps and formations
- **Parking management** — counting free spots, detecting illegal parking
- **Retail analytics** — customer flow mapping, queue measurement

In this notebook you will:
1. Build the scene — a synthetic road with cars and lane markings
2. Define the **ROI trapezoid** — the ground region you want to flatten
3. Apply `getPerspectiveTransform` + `warpPerspective`
4. Calibrate the view — map pixels to real-world metres
5. Measure inter-vehicle distance in the bird's eye view
6. Detect and count objects in the top-down frame
7. Package everything into a reusable `bird_eye_view()` function

## How to work in this lab

1. Run cells in order — each builds on the previous.
2. Every cell produces a visible output. Understand each stage before moving on.
3. You can upload a real dashcam or surveillance photo in Cell 13 to test the full pipeline.
4. Tunable parameters are marked with `# <-- TUNE` comments — experiment with them.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def show(img, title="", figsize=(8, 5)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show2(img1, title1, img2, title2, figsize=(14, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, img, title in zip(axes, [img1, img2], [title1, title2]):
        if len(img.shape) == 2:
            ax.imshow(img, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Libraries ready.")

## Cell 4 — Build the Road Scene

### What this cell does
- Generates a synthetic road scene as seen from a dashboard camera
- Includes lane markings (dashed white lines), three vehicles, and road shoulder markings
- The perspective is realistic: lane lines converge toward a vanishing point

### Why synthetic?
A real dashcam image would work too (Cell 13 lets you upload one), but a synthetic scene gives us ground truth: we know exactly where the lane lines are, what the real-world distances are, and what the correct transformation should produce. That makes it easy to verify correctness.

In [ ]:
np.random.seed(7)
IMG_H, IMG_W = 480, 720

# --- Sky and road background ---
scene = np.zeros((IMG_H, IMG_W, 3), dtype=np.uint8)
scene[:IMG_H//2] = (135, 185, 220)       # sky (light blue)
scene[IMG_H//2:] = (85,  82,  78)        # road (dark asphalt)

# Horizon gradient — slightly lighter near the horizon
for i in range(30):
    y = IMG_H // 2 + i
    brightness = int(85 + (30 - i) * 1.2)
    scene[y, :] = np.clip([brightness]*3, 0, 255)

# --- Perspective lane lines ---
# Vanishing point (where parallel lines meet)
VP = (IMG_W // 2, IMG_H // 2 - 10)

# Real-world lane layout (3 lanes, 3.5m each)
# At the bottom of image, lanes are spread wide; converge to VP at top
BOTTOM_Y = IMG_H - 1
lane_x_bottom = [120, 260, 400, 540, 600]   # 5 lines bounding 3 lanes + shoulders

def perspective_x(x_bottom, y, vp_x=VP[0], vp_y=VP[1], bottom_y=BOTTOM_Y):
    """Compute x-coordinate at height y given bottom x and vanishing point."""
    t = (y - bottom_y) / (vp_y - bottom_y)
    return int(x_bottom + t * (vp_x - x_bottom))

# Draw lane dividers
for x_b in lane_x_bottom:
    pts = [(perspective_x(x_b, y), y) for y in range(IMG_H//2, IMG_H, 1)]
    for i in range(len(pts)-1):
        cv2.line(scene, pts[i], pts[i+1], (210, 210, 210), 2)

# Dashed centre lines
for x_b in [260, 400]:
    for seg_start in range(IMG_H//2 + 10, IMG_H, 40):
        seg_end = min(seg_start + 20, IMG_H)
        y1, y2 = seg_start, seg_end
        x1 = perspective_x(x_b, y1)
        x2 = perspective_x(x_b, y2)
        cv2.line(scene, (x1, y1), (x2, y2), (255, 255, 180), 3)

# --- Cars ---
# Each car: (lane_centre_x_bottom, y_bottom, width, height, color)
cars = [
    (190, 420, 90, 55, (60,  100, 200)),   # left lane — blue
    (330, 380, 80, 48, (220,  60,  60)),   # centre lane — red  (farther)
    (470, 350, 70, 40, (60,  180,  80)),   # right lane — green (farthest)
]

CAR_POSITIONS = []   # store for later use

for (cx, cy_bottom, cw, ch, color) in cars:
    # Scale width/height with perspective (smaller = farther)
    scale_factor = (cy_bottom - IMG_H//2) / (IMG_H//2)   # 0..1
    w2 = int(cw * scale_factor)
    h2 = int(ch * scale_factor)
    x1, y1 = cx - w2//2, cy_bottom - h2
    x2, y2 = cx + w2//2, cy_bottom
    cv2.rectangle(scene, (x1, y1), (x2, y2), color, -1)
    # Windshield
    ww = max(1, w2 - 10)
    cv2.rectangle(scene, (x1+5, y1+4), (x1+5+ww, y1+4+max(1, h2//3)),
                  (200, 220, 240), -1)
    CAR_POSITIONS.append((cx, cy_bottom, w2, h2))

# --- Road markings ---
# Solid white shoulder lines
for x_b in [120, 600]:
    pts = [(perspective_x(x_b, y), y) for y in range(IMG_H//2, IMG_H, 1)]
    for i in range(len(pts)-1):
        cv2.line(scene, pts[i], pts[i+1], (240, 240, 240), 3)

# Distance markers on road (horizontal stripes)
for y in [420, 380, 350, 310]:
    x_l = perspective_x(120, y)
    x_r = perspective_x(600, y)
    cv2.line(scene, (x_l, y), (x_r, y), (120, 110, 100), 1)

show(scene, "Synthetic dashcam scene — road with 3 cars")

## Cell 5 — Define the ROI Trapezoid

### What this cell does
- Defines four source points that mark the ground region to flatten
- Visualises the trapezoid overlaid on the camera image

### The ROI trapezoid — key concept
The camera sees the flat road as a trapezoid (wide at the bottom, narrow at the top) because of perspective foreshortening. We choose 4 points that form a known rectangle in the **real world** but appear as a trapezoid in the **image**. These 4 points are the input to `getPerspectiveTransform`.

Choosing good ROI points:
- **Bottom corners** — wide, near the bottom edge, just inside the lane boundaries
- **Top corners** — narrower, near the horizon, aligned with the same lane boundaries
- The real-world shape they enclose should be a rectangle (equal-width road segment)

In production systems (Tesla Autopilot, Mobileye), these points are calibrated once per camera mount position using known road markings.

In [ ]:
# ROI trapezoid — these 4 points form a rectangle on the real road surface
# Order: TL, TR, BR, BL  (matching the destination rectangle order)
ROI_SRC = np.float32([
    [perspective_x(120, 290), 290],   # TL — left shoulder at ~40m ahead
    [perspective_x(600, 290), 290],   # TR — right shoulder at ~40m ahead
    [600, BOTTOM_Y - 10],             # BR — right shoulder near camera
    [120, BOTTOM_Y - 10],             # BL — left shoulder near camera
])

print("ROI source points (TL, TR, BR, BL):")
for name, pt in zip(["TL","TR","BR","BL"], ROI_SRC):
    print(f"  {name}: ({int(pt[0])}, {int(pt[1])})")

# Visualise
vis = scene.copy()
pts_draw = ROI_SRC.astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(vis, [pts_draw], True, (0, 255, 255), 2)
label_colors = [(0,220,0),(0,180,255),(0,0,255),(255,120,0)]
for pt, name, color in zip(ROI_SRC, ["TL","TR","BR","BL"], label_colors):
    cv2.circle(vis, tuple(pt.astype(int)), 8, color, -1)
    cv2.putText(vis, name, (int(pt[0])+10, int(pt[1])-6),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

# Also shade the ROI region
overlay = vis.copy()
cv2.fillPoly(overlay, [pts_draw], (0, 255, 255))
vis = cv2.addWeighted(vis, 0.85, overlay, 0.15, 0)

show(vis, "Camera view with ROI trapezoid (cyan region will be flattened)")

## Cell 6 — Define the Destination Rectangle and Warp

### What this cell does
- Defines the 4 destination points (a flat rectangle)
- Computes `getPerspectiveTransform` and applies `warpPerspective`
- Shows the result: the road viewed perfectly from above

### How to choose the output size
The output rectangle dimensions determine the **scale** of the bird's eye view.
- **Width** corresponds to the real-world width of the road inside the ROI (here: ~4.8m = 3 lanes × 1.6m)
- **Height** corresponds to the real-world length of road captured (here: ~40m ahead)

We'll calibrate the exact pixel-to-metre mapping in Cell 8. For now, choose output dimensions that give a roughly realistic aspect ratio: if the ROI is 4.8m wide and 40m long, the output should be roughly 5× taller than wide.

In [ ]:
# Output bird's eye view dimensions
BEV_W = 480   # pixels — corresponds to road width (left to right shoulder)
BEV_H = 600   # pixels — corresponds to road length (near to far)

# Destination: a clean rectangle
# Note: y=0 is the FAR end of the road (top of image = farthest point)
ROI_DST = np.float32([
    [0,       0      ],   # TL — far-left
    [BEV_W-1, 0      ],   # TR — far-right
    [BEV_W-1, BEV_H-1],  # BR — near-right
    [0,       BEV_H-1],  # BL — near-left
])

# Compute the perspective transform
M_bev = cv2.getPerspectiveTransform(ROI_SRC, ROI_DST)
M_bev_inv = cv2.getPerspectiveTransform(ROI_DST, ROI_SRC)  # for projecting back

print("H (perspective matrix):\n", np.round(M_bev, 4))

# Apply the warp
bev = cv2.warpPerspective(scene, M_bev, (BEV_W, BEV_H))

show2(scene, "Camera view (perspective)", bev, "Bird's eye view (top-down)")

## Cell 7 — Verify Lane Lines are Parallel

### What this cell does
- Runs Canny edge detection on the bird's eye view
- Uses `HoughLinesP` to detect straight line segments
- Checks that the detected lane lines are approximately vertical (parallel)

### Why this matters
In a perspective image, parallel road lanes converge toward the vanishing point. After a correct bird's eye transform, they should be **exactly parallel** again — vertical lines in the top-down image. Deviations indicate the ROI points need adjustment. This is the standard validation step when calibrating a new camera.

In [ ]:
# Detect lane lines in the bird's eye view
bev_gray   = cv2.cvtColor(bev, cv2.COLOR_BGR2GRAY)
bev_blur   = cv2.GaussianBlur(bev_gray, (5, 5), 0)
bev_edges  = cv2.Canny(bev_blur, 30, 100)

lines = cv2.HoughLinesP(bev_edges, rho=1, theta=np.pi/180,
                         threshold=50, minLineLength=80, maxLineGap=30)

vis_bev = bev.copy()
angles = []

if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        # Angle relative to vertical
        dx = x2 - x1
        dy = y2 - y1
        if dy == 0:
            continue
        angle_deg = abs(np.degrees(np.arctan2(dx, dy)))  # 0 = perfectly vertical
        angles.append(angle_deg)
        color = (0, 220, 0) if angle_deg < 5 else (0, 100, 255)
        cv2.line(vis_bev, (x1, y1), (x2, y2), color, 2)

show2(bev_edges, "Edges in BEV", vis_bev,
      "Lane lines — green=vertical (good), orange=tilted (bad)")

if angles:
    print(f"Detected {len(angles)} line segments")
    print(f"Deviation from vertical — mean: {np.mean(angles):.1f}°  max: {np.max(angles):.1f}°")
    print("Good calibration: mean < 3°. Poor calibration: mean > 10° — adjust ROI points.")
else:
    print("No lines detected.")

## Cell 8 — Pixel-to-Metre Calibration

### What this cell does
- Establishes the real-world scale: how many metres does one pixel represent?
- Uses a known real-world measurement (standard lane width) as reference
- Derives `px_per_m_x` and `px_per_m_y` independently (they may differ slightly)

### Calibration strategy
In the real world, a standard lane is **3.5m wide**. The ROI covers 3 lanes + 2 shoulders ≈ **10.5m total** from left shoulder to right shoulder. In the bird's eye view, that full width maps to `BEV_W = 480px`. So:

```
px_per_m_x = 480px / 10.5m = 45.7 px/m
1px = 1/45.7 = 0.022m = 2.2cm
```

For depth (y-axis), we use a known distance marker: the dashed lane lines have a known real-world repeat interval.

In [ ]:
# Real-world dimensions of the ROI region
ROAD_WIDTH_M  = 10.5   # left shoulder to right shoulder (3 lanes × 3.5m)
ROAD_LENGTH_M = 40.0   # approximate depth of ROI (camera to ~40m ahead)

# Pixel scale
px_per_m_x = BEV_W / ROAD_WIDTH_M
px_per_m_y = BEV_H / ROAD_LENGTH_M
m_per_px_x = 1.0 / px_per_m_x
m_per_px_y = 1.0 / px_per_m_y

print("=== Calibration ===")
print(f"BEV image:      {BEV_W} x {BEV_H} px")
print(f"ROI real size:  {ROAD_WIDTH_M}m wide  x  {ROAD_LENGTH_M}m long")
print(f"Scale X:        {px_per_m_x:.1f} px/m  ({m_per_px_x*100:.1f} cm/px)")
print(f"Scale Y:        {px_per_m_y:.1f} px/m  ({m_per_px_y*100:.1f} cm/px)")

# Draw a 10m reference scale bar on the BEV
bar_m   = 10
bar_px  = int(bar_m * px_per_m_y)
bar_x   = 20
bar_y1  = BEV_H - 20
bar_y2  = bar_y1 - bar_px

vis_calib = bev.copy()
cv2.line(vis_calib,  (bar_x, bar_y1), (bar_x, bar_y2), (0, 255, 255), 3)
cv2.line(vis_calib,  (bar_x-6, bar_y1), (bar_x+6, bar_y1), (0, 255, 255), 2)
cv2.line(vis_calib,  (bar_x-6, bar_y2), (bar_x+6, bar_y2), (0, 255, 255), 2)
cv2.putText(vis_calib, f"{bar_m}m", (bar_x+10, (bar_y1+bar_y2)//2 + 5),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

# Draw a 3.5m lane width reference
lane_px = int(3.5 * px_per_m_x)
lane_y  = BEV_H - 20
lane_x1 = int(0  * px_per_m_x)   # left shoulder x in BEV
lane_x2 = lane_x1 + lane_px
cv2.line(vis_calib, (lane_x1+5, lane_y), (lane_x2+5, lane_y), (255, 200, 0), 3)
cv2.putText(vis_calib, "3.5m", (lane_x1+8, lane_y-8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 200, 0), 2)

show(vis_calib, "BEV with calibration reference markers")

## Cell 9 — Measure Inter-Vehicle Distance

### What this cell does
- Projects the known car bottom-centre positions from camera view into the BEV
- Draws the projected positions on the top-down view
- Computes Euclidean distances between vehicles in real-world metres

### Why measure in BEV and not in the camera image?
In perspective, equal real-world distances appear shorter the farther they are. Two cars 10m apart at 30m range look closer together than two cars 10m apart at 10m range. In the bird's eye view, distances are **metrically consistent** — you can apply Euclidean geometry directly.

In [ ]:
# Project car bottom-centre points from camera image to BEV
# CAR_POSITIONS stores (cx, cy_bottom, w, h) from Cell 4

def project_to_bev(points_cam, H):
    """Project N points from camera image into BEV using homography H."""
    pts = np.array(points_cam, dtype=np.float32).reshape(-1, 1, 2)
    projected = cv2.perspectiveTransform(pts, H)
    return projected.reshape(-1, 2)

# Get bottom-centre of each car in camera coords
cam_pts = [(cx, cy_bottom) for (cx, cy_bottom, w, h) in CAR_POSITIONS]
bev_pts = project_to_bev(cam_pts, M_bev)

print("Car positions in BEV (pixels and metres from camera):")
car_names  = ["Blue (left)", "Red (centre)", "Green (right)"]
car_colors = [(200, 100, 50), (50, 50, 220), (50, 180, 50)]

vis_dist = bev.copy()
for i, (name, (bx, by), color) in enumerate(zip(car_names, bev_pts, car_colors)):
    dist_m = (BEV_H - by) * m_per_px_y   # distance from camera
    print(f"  {name}: BEV ({int(bx)}, {int(by)})  →  {dist_m:.1f}m ahead")
    cv2.circle(vis_dist, (int(bx), int(by)), 10, color, -1)
    cv2.putText(vis_dist, f"{dist_m:.0f}m",
                (int(bx)+12, int(by)+5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

# Measure distance between each pair of cars
print("\nInter-vehicle distances:")
for i in range(len(bev_pts)):
    for j in range(i+1, len(bev_pts)):
        dx_m = (bev_pts[j][0] - bev_pts[i][0]) * m_per_px_x
        dy_m = (bev_pts[j][1] - bev_pts[i][1]) * m_per_px_y
        d_m  = np.sqrt(dx_m**2 + dy_m**2)
        # Draw line
        p1 = tuple(bev_pts[i].astype(int))
        p2 = tuple(bev_pts[j].astype(int))
        cv2.line(vis_dist, p1, p2, (0, 220, 220), 1)
        mid = ((p1[0]+p2[0])//2, (p1[1]+p2[1])//2)
        cv2.putText(vis_dist, f"{d_m:.1f}m", mid,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 220, 220), 1)
        print(f"  {car_names[i]} — {car_names[j]}: {d_m:.1f}m")

show2(scene, "Camera view (perspective)", vis_dist,
      "BEV — car positions + inter-vehicle distances")

## Cell 10 — Detect and Count Vehicles in BEV

### What this cell does
- Detects vehicles in the bird's eye view using colour segmentation
- Draws bounding boxes and reports each vehicle's position in real-world coordinates
- Demonstrates why detection is easier in BEV than in perspective

### Why BEV helps detection
In a perspective image, vehicles at different distances appear at very different scales — a car 5m away looks 6× larger than one 30m away. Any fixed-size detector fails at some range. In the bird's eye view, all vehicles appear at the **same scale** regardless of distance, making size-based detection much more reliable.

In [ ]:
# Detect coloured vehicles in BEV (each car is a distinct colour)
bev_hsv = cv2.cvtColor(bev, cv2.COLOR_BGR2HSV)

# HSV ranges for each car colour
color_ranges = [
    ("Blue car",  np.array([100, 80, 60]),  np.array([130, 255, 255]), (200, 100,  50)),
    ("Red car",   np.array([  0, 80, 60]),  np.array([ 10, 255, 255]), ( 50,  50, 220)),
    ("Green car", np.array([ 45, 60, 60]),  np.array([ 85, 255, 255]), ( 50, 200,  50)),
]

vis_detect = bev.copy()
print("Detected vehicles in BEV:")

for name, lower, upper, draw_color in color_ranges:
    mask = cv2.inRange(bev_hsv, lower, upper)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  np.ones((5,5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9,9), np.uint8))

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts:
        area = cv2.contourArea(cnt)
        if area < 200:   # skip noise
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        cx_bev, cy_bev = x + w//2, y + h//2

        # Real-world position
        lane_x_m  = cx_bev * m_per_px_x
        dist_m    = (BEV_H - cy_bev) * m_per_px_y

        cv2.rectangle(vis_detect, (x, y), (x+w, y+h), draw_color, 2)
        cv2.circle(vis_detect, (cx_bev, cy_bev), 5, draw_color, -1)
        label = f"{name.split()[0]} {dist_m:.0f}m"
        cv2.putText(vis_detect, label, (x, y-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, draw_color, 1)
        print(f"  {name}: at {dist_m:.1f}m ahead, {lane_x_m:.1f}m from left edge")

show2(bev, "Raw BEV", vis_detect, "Detected vehicles in BEV")

## Cell 11 — Project BEV Detections Back to Camera View

### What this cell does
- Takes the bounding box centres found in the BEV
- Projects them back to the original camera image using the inverse homography `M_bev_inv`
- Draws the projected positions on the camera frame

### Why project back?
In a real ADAS system the camera feed is what the driver sees on a display. Detection happens in BEV (where geometry is easier), but the **warning annotations** must appear on the original camera image. The inverse homography maps BEV coordinates back to camera pixel coordinates.

In [ ]:
# Project BEV car positions back to camera image
bev_car_pts = bev_pts   # from Cell 9 — detected in BEV
cam_projected = project_to_bev(bev_car_pts, M_bev_inv)

vis_cam = scene.copy()
car_labels  = ["Blue", "Red", "Green"]
car_draw    = [(200, 100, 50), (50, 50, 220), (50, 200, 50)]

for (name, pt_bev, pt_cam, color) in zip(car_labels, bev_pts, cam_projected, car_draw):
    dist_m = (BEV_H - pt_bev[1]) * m_per_px_y
    px, py = int(pt_cam[0]), int(pt_cam[1])

    # Draw circle and distance annotation on camera image
    cv2.circle(vis_cam, (px, py), 10, color, -1)
    cv2.circle(vis_cam, (px, py), 12, (255, 255, 255), 2)  # white ring

    # Warning box above the vehicle
    warn_color = (0, 60, 255) if dist_m < 12 else (0, 200, 255)
    label = f"{dist_m:.0f}m {'CLOSE!' if dist_m < 12 else 'OK'}"
    lx, ly = px - 35, py - 20
    cv2.rectangle(vis_cam, (lx-2, ly-14), (lx+80, ly+4), warn_color, -1)
    cv2.putText(vis_cam, label, (lx, ly),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

show2(vis_cam, "BEV detections projected back to camera view",
      vis_detect, "Source: detections in BEV")

## Cell 12 — Full Pipeline Function

### What this cell does
- Packages everything into `bird_eye_view(img, roi_src, bev_w, bev_h, road_width_m, road_length_m)`
- Returns the BEV image, calibration constants, and the homography matrices
- Handles any camera/road configuration via the parameters

### Parameters explained
| Parameter | Meaning |
|-----------|--------|
| `roi_src` | 4 points (TL,TR,BR,BL) in the camera image marking the flat ground ROI |
| `bev_w`, `bev_h` | Output BEV image dimensions in pixels |
| `road_width_m` | Real-world width of the ROI region in metres |
| `road_length_m` | Real-world depth (near to far) of the ROI region in metres |

In [ ]:
def bird_eye_view(img, roi_src, bev_w=480, bev_h=600,
                  road_width_m=10.5, road_length_m=40.0):
    """
    Transform a camera image into a calibrated bird's eye view.

    Parameters
    ----------
    img           : BGR camera image
    roi_src       : np.float32 array (4,2) — ROI corners [TL,TR,BR,BL] in camera coords
    bev_w, bev_h  : output BEV image size
    road_width_m  : real-world width of ROI (metres)
    road_length_m : real-world depth of ROI (metres)

    Returns
    -------
    bev       : bird's eye view image
    H         : perspective transform matrix (camera → BEV)
    H_inv     : inverse transform (BEV → camera)
    cal       : dict with 'px_per_m_x', 'px_per_m_y', 'm_per_px_x', 'm_per_px_y'
    """
    roi_dst = np.float32([
        [0,       0      ],
        [bev_w-1, 0      ],
        [bev_w-1, bev_h-1],
        [0,       bev_h-1],
    ])
    H     = cv2.getPerspectiveTransform(roi_src, roi_dst)
    H_inv = cv2.getPerspectiveTransform(roi_dst, roi_src)
    bev   = cv2.warpPerspective(img, H, (bev_w, bev_h))
    cal   = {
        'px_per_m_x': bev_w / road_width_m,
        'px_per_m_y': bev_h / road_length_m,
        'm_per_px_x': road_width_m / bev_w,
        'm_per_px_y': road_length_m / bev_h,
    }
    return bev, H, H_inv, cal


def measure_distance_bev(pt1_bev, pt2_bev, cal):
    """
    Euclidean distance between two BEV points in metres.
    pt1, pt2 : (x, y) pixel coords in the BEV image
    """
    dx_m = (pt2_bev[0] - pt1_bev[0]) * cal['m_per_px_x']
    dy_m = (pt2_bev[1] - pt1_bev[1]) * cal['m_per_px_y']
    return np.sqrt(dx_m**2 + dy_m**2)


def bev_to_camera(pts_bev, H_inv):
    """Project BEV pixel coords back to camera image."""
    pts = np.array(pts_bev, dtype=np.float32).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, H_inv).reshape(-1, 2)


# --- Test the pipeline ---
bev_result, H_r, H_inv_r, cal_r = bird_eye_view(
    scene, ROI_SRC, BEV_W, BEV_H, ROAD_WIDTH_M, ROAD_LENGTH_M
)

show2(scene, "Input camera frame", bev_result, "bird_eye_view() output")
print("Calibration:", {k: round(v, 3) for k, v in cal_r.items()})

## Cell 13 — Test on Your Own Photo

### What this cell does
- Lets you upload a real dashcam or surveillance photo
- Runs the full pipeline with manually specified ROI points

### How to find ROI points in your photo
1. Open your image in any image viewer and note the pixel coordinates of 4 ground points that form a rectangle in the real world (e.g., lane lines at two known depths).
2. Fill them in as `roi_src` below in TL, TR, BR, BL order.
3. Adjust `road_width_m` and `road_length_m` to match your scene.

### Tips for better results
- The ROI region should cover **flat ground only** — avoid including sky or elevated objects
- Use lane markings or floor tiles as reference points — they have known real-world spacing
- Start with the road directly ahead, not extreme left/right angles

In [ ]:
try:
    from google.colab import files
    print("Upload a dashcam or road image:")
    uploaded = files.upload()
    if uploaded:
        path = list(uploaded.keys())[0]
        user_img = cv2.imread(path)
        if user_img is None:
            print("Could not read the file.")
        else:
            uh, uw = user_img.shape[:2]
            show(user_img, f"Uploaded: {path}  ({uw}x{uh})")
            print(f"Image size: {uw}x{uh}")
            print("\nDefine your ROI points below (TL, TR, BR, BL) based on the image.")

            # --- FILL IN YOUR ROI POINTS ---
            # Example for a typical 1280x720 dashcam image:
            user_roi = np.float32([
                [uw*0.42, uh*0.60],   # TL — adjust to your image
                [uw*0.58, uh*0.60],   # TR
                [uw*0.85, uh*0.90],   # BR
                [uw*0.15, uh*0.90],   # BL
            ])

            # Visualise the ROI
            vis_roi = user_img.copy()
            cv2.polylines(vis_roi, [user_roi.astype(np.int32)], True, (0,255,255), 2)
            for pt in user_roi:
                cv2.circle(vis_roi, tuple(pt.astype(int)), 8, (0,255,255), -1)
            show(vis_roi, "Your ROI (adjust user_roi coordinates if needed)")

            bev_user, _, _, cal_user = bird_eye_view(
                user_img, user_roi,
                bev_w=480, bev_h=600,
                road_width_m=10.5, road_length_m=40.0
            )
            show2(user_img, "Original", bev_user, "Bird's eye view")
    else:
        print("No file uploaded. Pipeline demonstrated on synthetic scene above.")
except Exception as e:
    print(f"Upload not available ({e}). Run in Google Colab to upload real images.")

## Summary

In this notebook you built a complete bird's eye view pipeline:

- **Scene setup** — synthetic dashcam view with perspective-correct lane lines and vehicles
- **ROI trapezoid** — 4 ground points that form a real-world rectangle, appear as a trapezoid in camera
- **getPerspectiveTransform + warpPerspective** — flattens the road to a top-down view
- **Validation** — lane lines should be vertical (parallel) after correct transform
- **Calibration** — pixel-to-metre scale from known real-world road dimensions
- **Distance measurement** — Euclidean distance between vehicles in real-world metres
- **Object detection in BEV** — colour segmentation, easier than in perspective
- **Back-projection** — inverse homography maps BEV detections to the camera frame
- **Reusable function** — `bird_eye_view()` + `measure_distance_bev()` + `bev_to_camera()`

**Where to go next:**
- Replace manual ROI with automatic lane detection (Hough lines on the perspective image)
- Use YOLO or HOG to detect vehicles in BEV instead of colour segmentation
- Build a speed estimator: track vehicle positions across frames, multiply by metres-per-pixel and FPS